In [1]:

import requests
import pandas as pd
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print("=== IMPORTATION GBFS FRANCE ===")

# 1. Vérification Lakehouse
base = Path("/lakehouse/default/Files")

if not base.exists():
    raise RuntimeError("Lakehouse non connecté")

destination = base / "bronze" / "gbfs_catalog_france"
destination.mkdir(parents=True, exist_ok=True)

# 2. Source officielle du catalogue MobilityData
url = (
    "https://raw.githubusercontent.com/"
    "MobilityData/gbfs/master/systems.csv"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

# 3. Conservation de la source brute
raw_path = destination / "systems.csv"
raw_path.write_bytes(response.content)

sha256 = hashlib.sha256(response.content).hexdigest()

print("Catalogue brut téléchargé")
print("SHA256 :", sha256)

# 4. Chargement du catalogue
df = pd.read_csv(
    raw_path,
    dtype=str,
    encoding="utf-8-sig"
).fillna("")

df.columns = df.columns.str.strip()

required = [
    "Country Code",
    "Name",
    "Location",
    "System ID",
    "Auto-Discovery URL"
]

missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Colonnes absentes : {missing}")

print("Réseaux mondiaux :", len(df))

# 5. Filtrage France + DROM
country_codes = ["FR", "GP", "MQ", "GF", "RE", "YT"]

france = df[
    df["Country Code"]
    .str.strip()
    .str.upper()
    .isin(country_codes)
].copy()

# 6. Vérification des URLs
france = france[
    france["Auto-Discovery URL"]
    .str.strip()
    .str.startswith(("https://", "http://"))
].copy()

france = france.drop_duplicates(
    subset=["System ID", "Auto-Discovery URL"]
)

# 7. Informations de provenance
france["catalog_source"] = "MobilityData"
france["collected_at_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

# 8. Sauvegarde dans Bronze
output = destination / "gbfs_france_catalog.csv"
france.to_csv(output, index=False)

# 9. Identification des réseaux Citiz
citiz = france[
    france["Name"].str.contains(
        "citiz", case=False, na=False
    )
].copy()

# 10. Résultats
print("\n=== RÉSULTATS ===")
print("Réseaux GBFS France :", len(france))
print("Réseaux Citiz :", len(citiz))
print("Fichier :", output)

print("\n=== RÉSEAUX PAR CODE PAYS ===")
display(
    france.groupby("Country Code")
    .size()
    .reset_index(name="nombre_reseaux")
)

print("\n=== RÉSEAUX CITIZ ===")
display(
    citiz[[
        "Name",
        "Location",
        "Auto-Discovery URL"
    ]]
)

print("\n=== APERÇU DES RÉSEAUX FRANÇAIS ===")
display(
    france[[
        "Name",
        "Location",
        "Country Code",
        "Auto-Discovery URL"
    ]].head(20)
)


In [2]:

import requests
import pandas as pd
import json
import re
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)

# ============================================
# 1. CONFIGURATION
# ============================================

BASE = Path("/lakehouse/default/Files")

CATALOG = (
    BASE / "bronze/gbfs_catalog_france/"
    "gbfs_france_catalog.csv"
)

OUTPUT = BASE / "bronze/gbfs_citiz"
OUTPUT.mkdir(parents=True, exist_ok=True)

catalog = pd.read_csv(CATALOG, dtype=str).fillna("")

citiz = catalog[
    catalog["Name"].str.contains(
        "Citiz", case=False, na=False
    )
].copy()

print("Réseaux Citiz détectés :", len(citiz))

session = requests.Session()
session.headers.update({
    "User-Agent": "MobilityIntelligenceHackathon/1.0",
    "Accept": "application/json"
})

timestamp = datetime.now(timezone.utc)
snapshot = timestamp.strftime("%Y%m%dT%H%M%SZ")

rows = []
reports = []


# ============================================
# 2. FONCTIONS UTILITAIRES
# ============================================

def get_json(url):
    response = session.get(
        url, timeout=(15, 40)
    )
    response.raise_for_status()
    return response.json()


def get_feeds(discovery):
    data = discovery.get("data", {})

    # GBFS 3.0
    if isinstance(data.get("feeds"), list):
        return {
            f["name"]: f["url"]
            for f in data["feeds"]
        }

    # Versions plus anciennes
    for value in data.values():
        if (
            isinstance(value, dict)
            and isinstance(value.get("feeds"), list)
        ):
            return {
                f["name"]: f["url"]
                for f in value["feeds"]
            }

    return {}


def station_name(value):
    # GBFS v2 : texte simple
    if isinstance(value, str):
        return value

    # GBFS v3 : traductions
    if isinstance(value, list):
        for item in value:
            if item.get("language") == "fr":
                return item.get("text", "")

        if value:
            return value[0].get("text", "")

    return ""


def valid_coordinates(lat, lon):
    try:
        lat = float(lat)
        lon = float(lon)

        if -90 <= lat <= 90 and -180 <= lon <= 180:
            return lat, lon
    except (TypeError, ValueError):
        pass

    return None


# ============================================
# 3. TÉLÉCHARGER LES RÉSEAUX CITIZ
# ============================================

for _, network in citiz.iterrows():

    name = network["Name"]
    system_id = network["System ID"]
    url = network["Auto-Discovery URL"]

    print(f"\n=== {name} ===")

    safe_id = re.sub(
        r"[^a-zA-Z0-9_-]", "_", system_id
    )

    folder = OUTPUT / snapshot / safe_id
    folder.mkdir(parents=True, exist_ok=True)

    try:
        # Lire le catalogue du réseau
        discovery = get_json(url)

        feeds = get_feeds(discovery)

        (folder / "gbfs.json").write_text(
            json.dumps(discovery, ensure_ascii=False),
            encoding="utf-8"
        )

        print("Flux disponibles :", list(feeds))

        if "station_information" not in feeds:
            reports.append({
                "network": name,
                "status": "NO_STATION_INFORMATION",
                "stations": 0,
                "error": ""
            })
            continue

        # Informations des stations
        station_data = get_json(
            feeds["station_information"]
        )

        (folder / "station_information.json").write_text(
            json.dumps(station_data, ensure_ascii=False),
            encoding="utf-8"
        )

        stations = (
            station_data.get("data", {})
            .get("stations", [])
        )

        # Disponibilité actuelle, si publiée
        statuses = {}

        if "station_status" in feeds:
            try:
                status_data = get_json(
                    feeds["station_status"]
                )

                (folder / "station_status.json").write_text(
                    json.dumps(status_data, ensure_ascii=False),
                    encoding="utf-8"
                )

                statuses = {
                    str(s["station_id"]): s
                    for s in status_data.get(
                        "data", {}
                    ).get("stations", [])
                }

            except Exception as e:
                print("Statut indisponible :", str(e))

        count = 0

        for station in stations:

            coordinates = valid_coordinates(
                station.get("lat"),
                station.get("lon")
            )

            if coordinates is None:
                continue

            lat, lon = coordinates
            station_id = str(
                station.get("station_id", "")
            )

            if not station_id:
                continue

            status = statuses.get(station_id, {})

            available = status.get(
                "num_vehicles_available"
            )

            rows.append({
                "station_key": (
                    f"{system_id}:{station_id}"
                ),
                "station_id": station_id,
                "station_name": station_name(
                    station.get("name")
                ),
                "latitude": lat,
                "longitude": lon,
                "capacity": station.get("capacity"),
                "available_vehicles": available,
                "operator": name,
                "system_id": system_id,
                "city_catalog": network["Location"],
                "source": "GBFS_CITIZ",
                "collected_at_utc": (
                    timestamp.isoformat()
                )
            })

            count += 1

        reports.append({
            "network": name,
            "status": "OK",
            "stations": count,
            "error": ""
        })

        print("Stations récupérées :", count)

    except Exception as e:

        print("Erreur :", str(e))

        reports.append({
            "network": name,
            "status": "ERROR",
            "stations": 0,
            "error": str(e)
        })


# ============================================
# 4. ENREGISTRER LE RAPPORT
# ============================================

report_df = pd.DataFrame(reports)

report_path = OUTPUT / snapshot / "import_report.csv"
report_df.to_csv(report_path, index=False)

print("\n=== RAPPORT NATIONAL CITIZ ===")

display(report_df)

print("Réseaux traités :", len(reports))
print("Stations obtenues :", len(rows))


# ============================================
# 5. CRÉER LA TABLE SILVER
# ============================================

if not rows:
    print("Aucune station valide récupérée.")
    print("Table Silver non modifiée.")
else:

    schema = StructType([
        StructField("station_key", StringType()),
        StructField("station_id", StringType()),
        StructField("station_name", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
        StructField("capacity", IntegerType()),
        StructField("available_vehicles", IntegerType()),
        StructField("operator", StringType()),
        StructField("system_id", StringType()),
        StructField("city_catalog", StringType()),
        StructField("source", StringType()),
        StructField("collected_at_utc", StringType())
    ])

    for row in rows:
        for field in ["capacity", "available_vehicles"]:
            try:
                value = row[field]
                row[field] = (
                    int(value) if value is not None else None
                )
            except (TypeError, ValueError):
                row[field] = None

    spark_df = spark.createDataFrame(
        rows, schema=schema
    )

    # Contrôle des identifiants
    duplicates = (
        spark_df.groupBy("station_key")
        .count()
        .filter("count > 1")
        .count()
    )

    if duplicates:
        raise ValueError(
            f"{duplicates} identifiants de station dupliqués"
        )

    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.france_citiz_stations")
    )

    print("\nTABLE CRÉÉE : silver.france_citiz_stations")
    print("Stations Citiz :", spark_df.count())

    display(
        spark_df.select(
            "station_name",
            "operator",
            "city_catalog",
            "latitude",
            "longitude",
            "available_vehicles"
        ).limit(20)
    )


In [3]:

import pandas as pd
import requests
import json
import re
from pathlib import Path
from datetime import datetime, timezone

# ===========================================
# CONFIGURATION
# ===========================================

base = Path("/lakehouse/default/Files")

catalogue = pd.read_csv(
    base / "bronze/gbfs_catalog_france/"
           "gbfs_france_catalog.csv",
    dtype=str
).fillna("")

destination = base / "bronze/gbfs_shared_mobility"
destination.mkdir(parents=True, exist_ok=True)

# Réseaux pilotes : vélos + trottinettes
pilotes = {
    "Velib_Paris": "Vélib",
    "Velhop_Strasbourg": "Vélhop",
    "Bird_Ajaccio": "Bird Ajaccio"
}

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

session = requests.Session()
session.headers.update({
    "User-Agent": "MobilityIntelligence-Hackathon/1.0"
})

# ===========================================
# FONCTIONS
# ===========================================

def fetch_json(url):
    response = session.get(url, timeout=(15, 45))
    response.raise_for_status()
    return response.json()


def discover_feeds(payload):
    data = payload.get("data", {})

    if isinstance(data.get("feeds"), list):
        feed_list = data["feeds"]
    else:
        feed_list = []
        for entry in data.values():
            if isinstance(entry, dict):
                if isinstance(entry.get("feeds"), list):
                    feed_list = entry["feeds"]
                    break

    return {
        feed["name"]: feed["url"]
        for feed in feed_list
        if "name" in feed and "url" in feed
    }


# ===========================================
# TEST DES TROIS RÉSEAUX
# ===========================================

rapport = []

for label, search_name in pilotes.items():

    selection = catalogue[
        catalogue["Name"].str.contains(
            search_name,
            case=False,
            regex=False,
            na=False
        )
    ]

    if selection.empty:
        print("Réseau absent :", search_name)
        rapport.append({
            "reseau": label,
            "statut": "ABSENT_CATALOGUE",
            "stations": 0,
            "vehicules": 0,
            "types": ""
        })
        continue

    network = selection.iloc[0]
    url = network["Auto-Discovery URL"]

    folder = destination / timestamp / label
    folder.mkdir(parents=True, exist_ok=True)

    try:
        print("\n==========================")
        print("Réseau :", network["Name"])

        discovery = fetch_json(url)
        feeds = discover_feeds(discovery)

        (folder / "gbfs.json").write_text(
            json.dumps(discovery, ensure_ascii=False),
            encoding="utf-8"
        )

        print("Flux disponibles :", list(feeds))

        results = {}
        errors = {}

        for feed_name in [
            "vehicle_types",
            "station_information",
            "station_status",
            "vehicle_status",
            "free_bike_status"
        ]:
            if feed_name not in feeds:
                continue

            try:
                payload = fetch_json(feeds[feed_name])

                results[feed_name] = payload

                (folder / f"{feed_name}.json").write_text(
                    json.dumps(payload, ensure_ascii=False),
                    encoding="utf-8"
                )

            except Exception as e:
                errors[feed_name] = str(e)
                print("Flux indisponible :", feed_name, e)

        # Informations sur les stations
        station_info = results.get(
            "station_information", {}
        )

        stations = (
            station_info.get("data", {})
            .get("stations", [])
        )

        # Véhicules sans station fixe
        vehicle_feed = (
            results.get("vehicle_status")
            or results.get("free_bike_status", {})
        )

        vehicle_data = vehicle_feed.get("data", {})

        vehicles = vehicle_data.get(
            "vehicles",
            vehicle_data.get("bikes", [])
        )

        # Types de véhicules
        type_info = results.get(
            "vehicle_types", {}
        )

        vehicle_types = (
            type_info.get("data", {})
            .get("vehicle_types", [])
        )

        form_factors = sorted({
            str(t.get("form_factor"))
            for t in vehicle_types
            if t.get("form_factor")
        })

        print("Stations :", len(stations))
        print("Véhicules individuels :", len(vehicles))
        print("Types déclarés :", form_factors)
        print("Erreurs des flux :", errors)

        rapport.append({
            "reseau": network["Name"],
            "statut": "OK" if not errors else "PARTIEL",
            "stations": len(stations),
            "vehicules": len(vehicles),
            "types": ", ".join(form_factors)
        })

    except Exception as e:
        print("Erreur réseau :", str(e))

        rapport.append({
            "reseau": label,
            "statut": "ERROR",
            "stations": 0,
            "vehicules": 0,
            "types": ""
        })

# ===========================================
# RÉSULTATS
# ===========================================

report = pd.DataFrame(rapport)

report.to_csv(
    destination / timestamp / "pilot_report.csv",
    index=False
)

print("\n=== RAPPORT GBFS PILOTE ===")
display(report)

print("\nFichiers enregistrés dans :", destination)


In [5]:

import pandas as pd
import requests
import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ==========================================
# 1. CONFIGURATION
# ==========================================

BATCH_INDEX = 1
BATCH_SIZE = 20

BASE = Path("/lakehouse/default/Files")

CATALOG_PATH = (
    BASE / "bronze/gbfs_catalog_france/"
    "gbfs_france_catalog.csv"
)

OUTPUT = BASE / "bronze/gbfs_shared_mobility/national"
OUTPUT.mkdir(parents=True, exist_ok=True)

catalog = pd.read_csv(
    CATALOG_PATH, dtype=str
).fillna("")

# Citiz a déjà son propre pipeline
networks = catalog[
    ~catalog["Name"].str.contains(
        "Citiz", case=False, na=False
    )
].copy()

networks = networks.sort_values(
    ["System ID", "Name"]
).reset_index(drop=True)

total = len(networks)
start = BATCH_INDEX * BATCH_SIZE
end = min(start + BATCH_SIZE, total)

batch = networks.iloc[start:end]

if batch.empty:
    raise ValueError("Lot vide : vérifier BATCH_INDEX")

print("=== INGESTION NATIONALE GBFS ===")
print("Réseaux à examiner :", total)
print("Lot :", BATCH_INDEX)
print("Réseaux de ce lot :", len(batch))
print("Positions :", start, "à", end - 1)

snapshot = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

batch_folder = OUTPUT / f"batch_{BATCH_INDEX:03d}_{snapshot}"
batch_folder.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. CONNEXION HTTP AVEC RETRY
# ==========================================

session = requests.Session()

retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

session.headers.update({
    "User-Agent": "MobilityIntelligenceHackathon/1.0",
    "Accept": "application/json"
})


def fetch_json(url):
    if not url.startswith(("https://", "http://")):
        raise ValueError("URL invalide")

    response = session.get(
        url,
        timeout=(15, 45)
    )

    response.raise_for_status()
    return response.json()


def discover_feeds(payload):
    data = payload.get("data", {})

    if not isinstance(data, dict):
        return {}

    feeds = data.get("feeds")

    if not isinstance(feeds, list):
        for value in data.values():
            if isinstance(value, dict):
                candidate = value.get("feeds")
                if isinstance(candidate, list):
                    feeds = candidate
                    break

    if not isinstance(feeds, list):
        return {}

    return {
        f["name"]: f["url"]
        for f in feeds
        if f.get("name") and f.get("url")
    }


def save_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)


# ==========================================
# 3. INGESTION DES RÉSEAUX
# ==========================================

reports = []

feed_names = [
    "system_information",
    "vehicle_types",
    "station_information",
    "station_status",
    "vehicle_status",
    "free_bike_status"
]

for index, network in batch.iterrows():

    name = network["Name"]
    url = network["Auto-Discovery URL"]
    system_id = network["System ID"]

    safe_id = re.sub(
        r"[^a-zA-Z0-9_-]", "_", system_id
    )

    suffix = hashlib.sha256(
        url.encode("utf-8")
    ).hexdigest()[:10]

    folder = batch_folder / f"{safe_id}_{suffix}"
    folder.mkdir(parents=True, exist_ok=True)

    print(f"\n=== {name} ===")

    report = {
        "network": name,
        "system_id": system_id,
        "url": url,
        "status": "ERROR",
        "stations": 0,
        "vehicles": 0,
        "vehicle_types": "",
        "error": ""
    }

    try:
        discovery = fetch_json(url)

        save_json(folder / "gbfs.json", discovery)

        feeds = discover_feeds(discovery)

        if not feeds:
            raise ValueError(
                "Aucun flux GBFS découvert"
            )

        data = {}
        feed_errors = []

        for feed_name in feed_names:

            if feed_name not in feeds:
                continue

            try:
                payload = fetch_json(feeds[feed_name])

                save_json(
                    folder / f"{feed_name}.json",
                    payload
                )

                data[feed_name] = payload

            except Exception as e:
                feed_errors.append(
                    f"{feed_name}: {str(e)[:150]}"
                )

        stations = (
            data.get("station_information", {})
            .get("data", {})
            .get("stations", [])
        )

        free_data = (
            data.get("vehicle_status")
            or data.get("free_bike_status")
            or {}
        ).get("data", {})

        vehicles = free_data.get(
            "vehicles",
            free_data.get("bikes", [])
        )

        types = (
            data.get("vehicle_types", {})
            .get("data", {})
            .get("vehicle_types", [])
        )

        form_factors = sorted({
            str(t.get("form_factor"))
            for t in types
            if t.get("form_factor")
        })

        report["stations"] = len(stations)
        report["vehicles"] = len(vehicles)
        report["vehicle_types"] = ", ".join(form_factors)

        if not data:
            report["status"] = "NO_USABLE_FEED"
        elif feed_errors:
            report["status"] = "PARTIAL"
        else:
            report["status"] = "OK"

        report["error"] = " | ".join(feed_errors)

        print("Stations :", len(stations))
        print("Véhicules :", len(vehicles))
        print("Types :", form_factors)
        print("Statut :", report["status"])

    except Exception as e:
        report["error"] = str(e)[:500]
        print("Échec :", report["error"])

    reports.append(report)

    # Sauvegarde progressive du rapport
    pd.DataFrame(reports).to_csv(
        batch_folder / "import_report.csv",
        index=False
    )

# ==========================================
# 4. BILAN
# ==========================================

result = pd.DataFrame(reports)

print("\n=== BILAN DU LOT GBFS ===")
print("Réseaux examinés :", len(result))
print("Réseaux OK :", (result["status"] == "OK").sum())
print(
    "Réseaux partiels :",
    (result["status"] == "PARTIAL").sum()
)
print(
    "Réseaux en échec :",
    result["status"].isin(
        ["ERROR", "NO_USABLE_FEED"]
    ).sum()
)

print("Stations publiées :", result["stations"].sum())
print("Véhicules publiés :", result["vehicles"].sum())

display(
    result[[
        "network",
        "status",
        "stations",
        "vehicles",
        "vehicle_types",
        "error"
    ]]
)

print("\nFichiers enregistrés dans :", batch_folder)


In [6]:

import pandas as pd
import requests
import json
import re
import hashlib
import time
from pathlib import Path
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ==========================================
# CONFIGURATION
# ==========================================

START_INDEX = 40  # Les lots 0 et 1 sont terminés
BATCH_SIZE = 20

BASE = Path("/lakehouse/default/Files")

CATALOG = (
    BASE / "bronze/gbfs_catalog_france/"
    "gbfs_france_catalog.csv"
)

OUTPUT = (
    BASE / "bronze/gbfs_shared_mobility/national"
)

OUTPUT.mkdir(parents=True, exist_ok=True)

# ==========================================
# CATALOGUE
# ==========================================

catalog = pd.read_csv(
    CATALOG, dtype=str
).fillna("")

networks = catalog[
    ~catalog["Name"].str.contains(
        "Citiz", case=False, na=False
    )
].copy()

networks = networks.sort_values(
    ["System ID", "Name"]
).reset_index(drop=True)

print("=== IMPORT NATIONAL AUTOMATIQUE ===")
print("Réseaux totaux :", len(networks))
print("Déjà examinés :", START_INDEX)
print("Réseaux restants :", len(networks) - START_INDEX)

# ==========================================
# SESSION HTTP
# ==========================================

session = requests.Session()

retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

session.headers.update({
    "User-Agent": "MobilityIntelligenceHackathon/1.0",
    "Accept": "application/json"
})

def fetch_json(url):
    if not url.startswith(("https://", "http://")):
        raise ValueError("URL invalide")

    response = session.get(
        url, timeout=(15, 45)
    )
    response.raise_for_status()
    return response.json()


def discover_feeds(payload):
    data = payload.get("data", {})

    if not isinstance(data, dict):
        return {}

    feeds = data.get("feeds")

    if not isinstance(feeds, list):
        for value in data.values():
            if isinstance(value, dict):
                candidate = value.get("feeds")
                if isinstance(candidate, list):
                    feeds = candidate
                    break

    if not isinstance(feeds, list):
        return {}

    return {
        f["name"]: f["url"]
        for f in feeds
        if f.get("name") and f.get("url")
    }


def save_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)


# ==========================================
# TRAITEMENT AUTOMATIQUE
# ==========================================

FEEDS = [
    "system_information",
    "vehicle_types",
    "station_information",
    "station_status",
    "vehicle_status",
    "free_bike_status"
]

reports = []
start_time = time.monotonic()

for index in range(START_INDEX, len(networks)):

    network = networks.iloc[index]

    name = network["Name"]
    system_id = network["System ID"]
    url = network["Auto-Discovery URL"]

    batch_index = index // BATCH_SIZE

    safe_id = re.sub(
        r"[^a-zA-Z0-9_-]", "_", system_id
    )

    unique_id = hashlib.sha256(
        url.encode("utf-8")
    ).hexdigest()[:10]

    folder = (
        OUTPUT
        / f"batch_{batch_index:03d}_auto"
        / f"{safe_id}_{unique_id}"
    )

    folder.mkdir(parents=True, exist_ok=True)

    checkpoint = folder / "checkpoint.json"

    print(f"\n[{index + 1}/{len(networks)}] {name}")

    # Ne pas retélécharger les réseaux réussis
    if checkpoint.exists():
        previous = json.loads(
            checkpoint.read_text(encoding="utf-8")
        )
        if previous.get("status") == "OK":
            print("Déjà terminé : ignoré")
            reports.append(previous)
            continue

    report = {
        "index": index,
        "network": name,
        "system_id": system_id,
        "status": "ERROR",
        "stations": 0,
        "vehicles": 0,
        "vehicle_types": "",
        "error": "",
        "collected_at_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    try:
        discovery = fetch_json(url)
        feeds = discover_feeds(discovery)

        if not feeds:
            raise ValueError("Aucun flux GBFS découvert")

        snapshot = datetime.now(
            timezone.utc
        ).strftime("%Y%m%dT%H%M%S%fZ")

        snapshot_folder = folder / "snapshots" / snapshot

        save_json(
            snapshot_folder / "gbfs.json",
            discovery
        )

        results = {}
        errors = []

        for feed_name in FEEDS:
            if feed_name not in feeds:
                continue

            try:
                data = fetch_json(feeds[feed_name])

                save_json(
                    snapshot_folder / f"{feed_name}.json",
                    data
                )

                results[feed_name] = data

            except Exception as e:
                errors.append(
                    f"{feed_name}: {str(e)[:120]}"
                )

        stations = (
            results.get("station_information", {})
            .get("data", {})
            .get("stations", [])
        )

        vehicle_feed = (
            results.get("vehicle_status")
            or results.get("free_bike_status")
            or {}
        )

        vehicle_data = vehicle_feed.get("data", {})

        vehicles = vehicle_data.get(
            "vehicles",
            vehicle_data.get("bikes", [])
        )

        types = (
            results.get("vehicle_types", {})
            .get("data", {})
            .get("vehicle_types", [])
        )

        form_factors = sorted({
            str(t.get("form_factor"))
            for t in types
            if t.get("form_factor")
        })

        report["stations"] = len(stations)
        report["vehicles"] = len(vehicles)
        report["vehicle_types"] = ", ".join(form_factors)
        report["snapshot_path"] = str(snapshot_folder)
        report["error"] = " | ".join(errors)

        if not results:
            report["status"] = "NO_USABLE_FEED"
        elif errors:
            report["status"] = "PARTIAL"
        else:
            report["status"] = "OK"

    except Exception as e:
        report["error"] = str(e)[:500]

    # Checkpoint après chaque réseau
    save_json(checkpoint, report)
    reports.append(report)

    print("Statut :", report["status"])
    print("Stations :", report["stations"])
    print("Véhicules :", report["vehicles"])

    # Sauvegarde progressive du rapport
    pd.DataFrame(reports).to_csv(
        OUTPUT / "automatic_import_report.csv",
        index=False
    )

    time.sleep(0.3)

# ==========================================
# RAPPORT FINAL
# ==========================================

result = pd.DataFrame(reports)

print("\n=== BILAN GLOBAL AUTOMATIQUE ===")

print("Réseaux du traitement :", len(result))

print(
    "Réseaux OK :",
    (result["status"] == "OK").sum()
)

print(
    "Réseaux partiels :",
    (result["status"] == "PARTIAL").sum()
)

print(
    "Réseaux en échec :",
    result["status"].isin(
        ["ERROR", "NO_USABLE_FEED"]
    ).sum()
)

print("Stations :", result["stations"].sum())
print("Véhicules :", result["vehicles"].sum())

print(
    "Durée (minutes) :",
    round((time.monotonic() - start_time) / 60, 2)
)

display(
    result[[
        "network",
        "status",
        "stations",
        "vehicles",
        "vehicle_types",
        "error"
    ]]
)

print("\nIMPORT AUTOMATIQUE TERMINÉ")


In [7]:

import json
import hashlib
import re
import pandas as pd

from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)

BASE = Path("/lakehouse/default/Files")
ROOT = BASE / "bronze/gbfs_shared_mobility/national"

CATALOG = BASE / (
    "bronze/gbfs_catalog_france/"
    "gbfs_france_catalog.csv"
)

print("=== NORMALISATION GBFS FRANCE ===")

# 1. Retrouver les réseaux du catalogue
catalog = pd.read_csv(CATALOG, dtype=str).fillna("")

network_map = {}

for _, r in catalog.iterrows():
    url = r["Auto-Discovery URL"]
    system_id = r["System ID"]

    safe_id = re.sub(
        r"[^a-zA-Z0-9_-]", "_", system_id
    )

    suffix = hashlib.sha256(
        url.encode("utf-8")
    ).hexdigest()[:10]

    network_map[f"{safe_id}_{suffix}"] = {
        "name": r["Name"],
        "system_id": system_id,
        "catalog_location": r["Location"]
    }

# 2. Rechercher les fichiers déjà téléchargés
files = list(ROOT.rglob("station_information.json"))

# Conserver le dernier instantané par réseau
latest = {}

for path in files:
    folder = path.parent

    if folder.parent.name == "snapshots":
        folder = folder.parent.parent

    if not folder.parent.name.startswith("batch_"):
        continue

    key = folder.name

    if (
        key not in latest
        or path.stat().st_mtime >
           latest[key].stat().st_mtime
    ):
        latest[key] = path

print("Fichiers trouvés :", len(files))
print("Réseaux avec stations :", len(latest))

rows = []
invalid = 0
duplicates = 0
errors = []
seen = set()

def localized_name(value):
    if isinstance(value, str):
        return value

    if isinstance(value, list):
        for entry in value:
            if isinstance(entry, dict):
                if entry.get("language") == "fr":
                    return str(entry.get("text", ""))

        if value and isinstance(value[0], dict):
            return str(value[0].get("text", ""))

    return ""

def safe_int(value):
    try:
        return int(value) if value is not None else None
    except (ValueError, TypeError):
        return None

# 3. Normaliser les stations
for network_key, path in latest.items():
    try:
        info = json.loads(
            path.read_text(encoding="utf-8")
        )

        stations = info.get(
            "data", {}
        ).get("stations", [])

        # Identifier les modes déclarés par le réseau
        type_path = path.parent / "vehicle_types.json"

        modes = set()

        if type_path.exists():
            types = json.loads(
                type_path.read_text(encoding="utf-8")
            ).get("data", {}).get("vehicle_types", [])

            modes = {
                str(v.get("form_factor"))
                for v in types
                if v.get("form_factor")
            }

        if len(modes) == 1:
            mode_hint = next(iter(modes))
        elif len(modes) > 1:
            mode_hint = "mixed"
        else:
            mode_hint = "unknown"

        network = network_map.get(network_key, {})

        for station in stations:
            sid = str(station.get("station_id") or "")

            try:
                lat = float(station["lat"])
                lon = float(station["lon"])
            except (KeyError, ValueError, TypeError):
                invalid += 1
                continue

            if (
                not sid
                or not -90 <= lat <= 90
                or not -180 <= lon <= 180
            ):
                invalid += 1
                continue

            station_key = f"{network_key}:{sid}"

            if station_key in seen:
                duplicates += 1
                continue

            seen.add(station_key)

            virtual = station.get("is_virtual_station")

            if virtual is True:
                station_kind = "virtual"
            elif virtual is False:
                station_kind = "non_virtual"
            else:
                station_kind = "unknown"

            rows.append({
                "station_key": station_key,
                "station_id": sid,
                "station_name": localized_name(
                    station.get("name")
                ),
                "latitude": lat,
                "longitude": lon,
                "operator": network.get("name", ""),
                "system_id": network.get("system_id", ""),
                "mode_hint": mode_hint,
                "declared_modes": ", ".join(sorted(modes)),
                "station_kind": station_kind,
                "capacity": safe_int(
                    station.get("capacity")
                ),
                "source": "GBFS",
                "source_file": str(path)
            })

    except Exception as e:
        errors.append({
            "network_key": network_key,
            "error": str(e)[:300]
        })

# 4. Contrôles
print("\n=== QUALITÉ ===")
print("Emplacements valides :", len(rows))
print("Coordonnées/ID invalides :", invalid)
print("Identifiants dupliqués :", duplicates)
print("Fichiers en erreur :", len(errors))

if not rows:
    raise RuntimeError("Aucune station exploitable")

# 5. Définir explicitement les types Spark
schema = StructType([
    StructField("station_key", StringType()),
    StructField("station_id", StringType()),
    StructField("station_name", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("operator", StringType()),
    StructField("system_id", StringType()),
    StructField("mode_hint", StringType()),
    StructField("declared_modes", StringType()),
    StructField("station_kind", StringType()),
    StructField("capacity", IntegerType()),
    StructField("source", StringType()),
    StructField("source_file", StringType())
])

sdf = spark.createDataFrame(rows, schema=schema)

# 6. Sauvegarder dans Silver
(
    sdf.withColumn(
        "processed_at_utc",
        F.current_timestamp()
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.france_gbfs_station_locations")
)

print(
    "\nTABLE CRÉÉE : "
    "silver.france_gbfs_station_locations"
)

# 7. Analyser les emplacements
print("\n=== TYPES D'EMPLACEMENTS ===")

display(
    sdf.groupBy("station_kind")
    .count()
    .orderBy(F.desc("count"))
)

print("\n=== MODES DÉCLARÉS ===")

display(
    sdf.groupBy("mode_hint")
    .count()
    .orderBy(F.desc("count"))
)

print("\n=== EXEMPLES ===")

display(
    sdf.select(
        "station_name",
        "operator",
        "mode_hint",
        "station_kind",
        "latitude",
        "longitude"
    ).limit(20)
)

if errors:
    print("\nFichiers à examiner :")
    display(pd.DataFrame(errors))


In [8]:
 
from pyspark.sql import functions as F

df = spark.table(
    "silver.france_gbfs_station_locations"
)

print("=== RÉPARTITION PAR MODE ===")

display(
    df.groupBy("mode_hint")
    .agg(F.count("*").alias("emplacements"))
    .orderBy(F.desc("emplacements"))
)

print("=== MODE ET TYPE DE STATION ===")

display(
    df.groupBy("mode_hint", "station_kind")
    .agg(F.count("*").alias("nombre"))
    .orderBy(F.desc("nombre"))
)

print("=== LES 15 PRINCIPAUX RÉSEAUX ===")

display(
    df.groupBy("operator")
    .agg(
        F.count("*").alias("emplacements"),
        F.countDistinct("station_id").alias(
            "ids_distincts"
        )
    )
    .orderBy(F.desc("emplacements"))
    .limit(15)
)


In [9]:

import json
import re
import hashlib
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, BooleanType
)

# =========================================
# 1. CONFIGURATION
# =========================================

BASE = Path("/lakehouse/default/Files")

ROOT = (
    BASE / "bronze/gbfs_shared_mobility/national"
)

CATALOG = (
    BASE / "bronze/gbfs_catalog_france/"
    "gbfs_france_catalog.csv"
)

print("=== NORMALISATION VÉHICULES GBFS ===")

catalog = pd.read_csv(CATALOG, dtype=str).fillna("")

network_map = {}

for _, r in catalog.iterrows():
    safe_id = re.sub(
        r"[^a-zA-Z0-9_-]", "_",
        r["System ID"]
    )

    suffix = hashlib.sha256(
        r["Auto-Discovery URL"].encode("utf-8")
    ).hexdigest()[:10]

    network_map[f"{safe_id}_{suffix}"] = {
        "name": r["Name"],
        "system_id": r["System ID"]
    }

# =========================================
# 2. RECHERCHER LES FICHIERS VÉHICULES
# =========================================

paths = (
    list(ROOT.rglob("vehicle_status.json"))
    + list(ROOT.rglob("free_bike_status.json"))
)

latest = {}

for path in paths:

    # Structure des lots automatiques
    if path.parent.parent.name == "snapshots":
        network_folder = path.parent.parent.parent
    else:
        network_folder = path.parent

    key = network_folder.name

    # Conserver le dernier instantané disponible
    if (
        key not in latest
        or path.stat().st_mtime >
           latest[key].stat().st_mtime
    ):
        latest[key] = path

print("Fichiers trouvés :", len(paths))
print("Réseaux avec flux véhicules :", len(latest))

# =========================================
# 3. CLASSIFICATION DES MODES
# =========================================

def classify_mode(form_factor):
    mapping = {
        "bicycle": "velo",
        "cargo_bicycle": "velo",
        "scooter_standing": "trottinette",
        "car": "voiture",
        "scooter": "scooter_indetermine",
        "scooter_seated": "deux_roues_motorise",
        "moped": "deux_roues_motorise"
    }

    return mapping.get(
        str(form_factor).lower(),
        "unknown"
    )


def valid_coordinates(vehicle):
    try:
        lat = float(vehicle.get("lat"))
        lon = float(vehicle.get("lon"))

        if (
            -90 <= lat <= 90
            and -180 <= lon <= 180
            and not (lat == 0 and lon == 0)
        ):
            return lat, lon
    except (ValueError, TypeError):
        pass

    return None


# =========================================
# 4. NORMALISER LES VÉHICULES
# =========================================

rows = []
invalid = 0
duplicates = 0
errors = []

seen = set()

for network_key, path in latest.items():

    try:
        payload = json.loads(
            path.read_text(encoding="utf-8")
        )

        data = payload.get("data", {})

        vehicles = data.get("vehicles")
        if vehicles is None:
            vehicles = data.get("bikes", [])

        types_file = path.parent / "vehicle_types.json"

        type_map = {}

        if types_file.exists():
            types_data = json.loads(
                types_file.read_text(encoding="utf-8")
            )

            for t in types_data.get(
                "data", {}
            ).get("vehicle_types", []):
                type_map[
                    str(t.get("vehicle_type_id", ""))
                ] = t.get("form_factor", "")

        network = network_map.get(network_key, {})

        collected_at = datetime.fromtimestamp(
            path.stat().st_mtime,
            timezone.utc
        ).isoformat()

        for v in vehicles:

            vid = str(
                v.get("vehicle_id")
                or v.get("bike_id")
                or ""
            )

            coords = valid_coordinates(v)

            if not vid or coords is None:
                invalid += 1
                continue

            vehicle_key = f"{network_key}:{vid}"

            if vehicle_key in seen:
                duplicates += 1
                continue

            seen.add(vehicle_key)

            type_id = str(
                v.get("vehicle_type_id") or ""
            )

            form_factor = type_map.get(
                type_id, ""
            )

            lat, lon = coords

            rows.append({
                "vehicle_key": vehicle_key,
                "vehicle_id": vid,
                "operator": network.get("name", ""),
                "system_id": network.get("system_id", ""),
                "vehicle_type_id": type_id,
                "form_factor": str(form_factor),
                "mode": classify_mode(form_factor),
                "latitude": lat,
                "longitude": lon,
                "is_reserved": v.get("is_reserved")
                    if isinstance(
                        v.get("is_reserved"), bool
                    ) else None,
                "is_disabled": v.get("is_disabled")
                    if isinstance(
                        v.get("is_disabled"), bool
                    ) else None,
                "collected_at_utc": collected_at,
                "source_file": str(path)
            })

    except Exception as e:
        errors.append({
            "network": network_key,
            "error": str(e)[:300]
        })

# =========================================
# 5. CONTRÔLES
# =========================================

print("\n=== QUALITÉ ===")
print("Véhicules géolocalisés :", len(rows))
print("Identifiants/positions invalides :", invalid)
print("Doublons internes :", duplicates)
print("Fichiers en erreur :", len(errors))

if not rows:
    raise RuntimeError(
        "Aucun véhicule géolocalisé trouvé"
    )

# =========================================
# 6. CRÉER LA TABLE SILVER
# =========================================

schema = StructType([
    StructField("vehicle_key", StringType()),
    StructField("vehicle_id", StringType()),
    StructField("operator", StringType()),
    StructField("system_id", StringType()),
    StructField("vehicle_type_id", StringType()),
    StructField("form_factor", StringType()),
    StructField("mode", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("is_reserved", BooleanType()),
    StructField("is_disabled", BooleanType()),
    StructField("collected_at_utc", StringType()),
    StructField("source_file", StringType())
])

sdf = spark.createDataFrame(rows, schema=schema)

(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.france_gbfs_vehicle_locations")
)

print(
    "\nTABLE CRÉÉE : "
    "silver.france_gbfs_vehicle_locations"
)

# =========================================
# 7. STATISTIQUES
# =========================================

print("\n=== RÉPARTITION DES VÉHICULES ===")

display(
    sdf.groupBy("mode")
    .count()
    .orderBy(F.desc("count"))
)

print("\n=== PRINCIPAUX OPÉRATEURS ===")

display(
    sdf.groupBy("operator", "mode")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

if errors:
    print("\n=== ERREURS ===")
    display(pd.DataFrame(errors))


In [10]:

from pyspark.sql import functions as F

df = spark.table(
    "silver.france_gbfs_vehicle_locations"
)

# 1. Filtrer Getaround
getaround = df.filter(
    F.lower(F.col("operator")).contains("getaround")
)

print("=== AUDIT GETAROUND ===")

print("Enregistrements :", getaround.count())

print(
    "Identifiants véhicule distincts :",
    getaround.select("vehicle_id").distinct().count()
)

print(
    "Flux géographiques :",
    getaround.select("system_id").distinct().count()
)

# 2. Rechercher les ID présents dans plusieurs flux
duplicates = (
    getaround.groupBy("vehicle_id")
    .agg(
        F.count("*").alias("occurrences"),
        F.countDistinct("system_id").alias("nb_flux")
    )
    .filter(F.col("nb_flux") > 1)
)

print(
    "Identifiants présents dans plusieurs flux :",
    duplicates.count()
)

print("\n=== EXEMPLES DE DOUBLONS POTENTIELS ===")

display(
    duplicates.orderBy(
        F.desc("nb_flux")
    ).limit(15)
)

# 3. Statuts de disponibilité
print("\n=== DISPONIBILITÉ ===")

display(
    df.groupBy("mode")
    .agg(
        F.count("*").alias("total"),
        F.sum(
            F.when(
                (F.col("is_reserved") == False) &
                (F.col("is_disabled") == False),
                1
            ).otherwise(0)
        ).alias("confirmes_non_reserves_non_desactives"),
        F.sum(
            F.when(
                F.col("is_reserved").isNull() |
                F.col("is_disabled").isNull(),
                1
            ).otherwise(0)
        ).alias("statut_incomplet")
    )
    .orderBy(F.desc("total"))
)


In [11]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=== DÉDOUBLONNAGE NATIONAL GBFS ===")

# 1. Lire la table originale
df = spark.table(
    "silver.france_gbfs_vehicle_locations"
)

# 2. Séparer Getaround des autres opérateurs
is_getaround = F.lower(
    F.coalesce(F.col("operator"), F.lit(""))
).startswith("getaround")

getaround = df.filter(is_getaround)
others = df.filter(~is_getaround)

print("Total initial :", df.count())
print("Getaround initial :", getaround.count())

# 3. Auditer les positions différentes d'un même ID
position_audit = (
    getaround.groupBy("vehicle_id")
    .agg(
        F.count("*").alias("occurrences"),
        F.countDistinct(
            F.struct(
                F.round("latitude", 5),
                F.round("longitude", 5)
            )
        ).alias("positions_distinctes")
    )
)

conflicts = position_audit.filter(
    F.col("positions_distinctes") > 1
).count()

print("IDs avec plusieurs positions :", conflicts)

# 4. Conserver une seule observation par véhicule
# L'observation la plus récente est privilégiée.
window = (
    Window.partitionBy("vehicle_id")
    .orderBy(
        F.col("collected_at_utc").desc_nulls_last(),
        F.col("source_file").desc_nulls_last(),
        F.col("vehicle_key").desc()
    )
)

getaround_unique = (
    getaround
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

# 5. Réunir avec les autres opérateurs
dedup = others.unionByName(getaround_unique)

initial_count = df.count()
final_count = dedup.count()

print("\n=== RÉSULTATS ===")
print("Avant :", initial_count)
print("Après :", final_count)
print("Enregistrements supprimés :",
      initial_count - final_count)

print("Getaround uniques :",
      getaround_unique.count())

# 6. Contrôle d'unicité Getaround
remaining_duplicates = (
    getaround_unique.groupBy("vehicle_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if remaining_duplicates:
    raise ValueError("Doublons Getaround restants")

# 7. Écrire une nouvelle table Silver
(
    dedup.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "silver.france_gbfs_vehicle_locations_dedup"
    )
)

print("\nTable nettoyée créée avec succès !")

# 8. Vérifier la répartition finale
display(
    dedup.groupBy("mode")
    .agg(F.count("*").alias("vehicules"))
    .orderBy(F.desc("vehicules"))
)

print("\n=== POSITIONS GETAROUND À EXAMINER ===")
display(
    position_audit.filter(
        F.col("positions_distinctes") > 1
    ).orderBy(
        F.desc("positions_distinctes")
    ).limit(10)
)
